# The Rest Is History — A Revolucao Francesa (EN + PT)
Transcreve **todos os 13 episodios** com **Whisper large-v3** (timestamps por palavra)
e traduz **EN->PT**. Saida: `segments.json` + `.srt` por episodio.

## Antes de rodar
1. Suba **`french_revolution_audios.zip`** para a raiz do seu Google Drive (MyDrive).
2. `Ambiente de execucao -> Alterar tipo -> GPU (T4)`.
3. `Ambiente de execucao -> Executar tudo` e autorize a montagem do Drive.

Leva ~1-2 h numa T4. Cada episodio e salvo no Drive assim que termina - se a
sessao cair, e so rodar de novo que ele pula os ja prontos. No fim baixa um zip.


## 1. Dependencias


In [ ]:
!pip -q install -U faster-whisper transformers sentencepiece sacremoses srt
import torch
assert torch.cuda.is_available(), 'Ative a GPU: Ambiente de execucao -> Alterar tipo -> GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Montar o Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive')
OUT   = DRIVE / 'fr_outputs'      # resultados (persistem no Drive)
OUT.mkdir(exist_ok=True)
print('saida:', OUT)


## 3. Descompactar os audios


In [ ]:
import zipfile, pathlib
ZIP = DRIVE / 'french_revolution_audios.zip'
assert ZIP.exists(), f'Suba o zip para {ZIP} antes de rodar.'
AUD = pathlib.Path('/content/audios'); AUD.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(AUD)
eps = sorted(p.parent.name for p in AUD.glob('ep*/audio.mp3'))
print(len(eps), 'episodios:'); [print(' ', e) for e in eps]


## 4. Carregar os modelos (Whisper large-v3 + tradutor EN->PT)


In [ ]:
from faster_whisper import WhisperModel
from transformers import MarianMTModel, MarianTokenizer

asr = WhisperModel('large-v3', device='cuda', compute_type='float16')

MT = 'Helsinki-NLP/opus-mt-tc-big-en-pt'
tok = MarianTokenizer.from_pretrained(MT)
mt  = MarianMTModel.from_pretrained(MT).to('cuda').eval()
print('modelos prontos')


## 5. Funcoes: transcrever, traduzir, gravar


In [ ]:
import json, datetime, srt, torch

TITLES = {"ep01_marie_antoinette": "Part 1 — Why Marie Antoinette Became the Most Hated Woman in France", "ep02_diamond_necklace": "Part 2 — The Diamond Necklace Scandal", "ep03_violence_begins": "Part 3 — The Violence Begins", "ep04_showdown_versailles": "Part 4 — Showdown in Versailles", "ep05_storming_bastille": "Part 5 — The Storming of the Bastille", "ep06_rights_of_man": "Part 6 — The Rights of Man", "ep07_women_evict_louis": "Part 7 — The Women That Evicted Louis XVI From Versailles", "ep08_royal_family_escapes": "Final Part — The Royal Family Escapes", "ep09_la_marseillaise": "S02E05 — La Marseillaise", "ep10_september_massacres": "The First Terror — The September Massacres", "ep11_monarchy_last_breath": "S03E03 — The Monarchy's Last Breath", "ep12_trial_execution_louis": "S03E04 — The Trial and Execution of Louis XVI", "ep13_unexpected_revolutionary": "S03E02 — The Most Unexpected Revolutionary"}

def transcribe(path):
    segs, info = asr.transcribe(str(path), language='en',
                                word_timestamps=True, vad_filter=True,
                                beam_size=5)
    out = []
    for s in segs:
        words = [{'w': w.word.strip(), 's': round(w.start, 3), 'e': round(w.end, 3)}
                 for w in (s.words or []) if w.word.strip()]
        out.append({'start': round(s.start, 3), 'end': round(s.end, 3),
                    'en': s.text.strip(), 'words': words})
    return out

def translate(texts, bs=32):
    res = []
    for i in range(0, len(texts), bs):
        batch = [(t or ' ') for t in texts[i:i+bs]]
        enc = tok(batch, return_tensors='pt', padding=True,
                  truncation=True, max_length=512).to('cuda')
        with torch.no_grad():
            gen = mt.generate(**enc, num_beams=4, max_length=512)
        res += tok.batch_decode(gen, skip_special_tokens=True)
    return res

def td(x): return datetime.timedelta(seconds=float(x))

def write_outputs(ep, segs):
    pt = translate([s['en'] for s in segs])
    data, su_en, su_pt = [], [], []
    for i, (s, p) in enumerate(zip(segs, pt)):
        s['i'] = i; s['pt'] = p.strip()
        data.append(s)
        su_en.append(srt.Subtitle(i+1, td(s['start']), td(s['end']), s['en']))
        su_pt.append(srt.Subtitle(i+1, td(s['start']), td(s['end']), p.strip()))
    d = OUT / ep; d.mkdir(exist_ok=True)
    meta = {'episode': ep, 'title': TITLES.get(ep, ep),
            'show': 'The Rest Is History', 'model': 'large-v3', 'n_segments': len(data)}
    with open(d/'segments.json','w',encoding='utf-8') as f:
        json.dump({'meta': meta, 'segments': data}, f, ensure_ascii=False, indent=1)
    with open(d/'transcript.en.srt','w',encoding='utf-8') as f: f.write(srt.compose(su_en))
    with open(d/'transcript.pt.srt','w',encoding='utf-8') as f: f.write(srt.compose(su_pt))


## 6. Processar todos (incremental - pula os ja feitos)


In [ ]:
import time
for ep in eps:
    if (OUT / ep / 'segments.json').exists():
        print(f'[skip] {ep}'); continue
    t0 = time.time()
    print(f'[asr ] {ep} ...', flush=True)
    segs = transcribe(AUD / ep / 'audio.mp3')
    write_outputs(ep, segs)
    print(f'[ ok ] {ep}: {len(segs)} segmentos em {(time.time()-t0)/60:.1f} min')
print('CONCLUIDO - resultados em', OUT)


## 7. Baixar os resultados (tambem ja estao no seu Drive em `fr_outputs/`)


In [ ]:
import shutil
shutil.make_archive('/content/fr_outputs', 'zip', str(OUT))
from google.colab import files
files.download('/content/fr_outputs.zip')
